# Friend DMLPA final Colab/Kaggle runner

This notebook is the final minimal runner for the friend's DMLPA idea. It is deliberately not the main paper claim. It trains PPO with a DMLPA-style feature extractor and exports reproducible artifacts.

Defaults:

- Scientific action contract: `thesis_factorized + thesis_strict`.
- Rich observation surface: `env_sdm_history_reward` with `v5`.
- DMLPA variant: `faithful`, preserving the original `Linear -> GELU -> Linear -> TransformerEncoder -> last token` design.

The positional-encoding version from the new notebook is included as `DMLPA_VARIANT = "positional_v1_2"`, but it is an ablation, not the faithful friend baseline.


In [ ]:
# =====================
# 0) Minimal run config
# =====================

RUN_PROFILE = "debug"  # debug or serious
CONTRACT_PROFILE = "scientific_thesis_factorized"  # scientific_thesis_factorized or legacy_box18
DMLPA_VARIANT = "faithful"  # faithful or positional_v1_2

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_DIR_COLAB = "/content/scresia"
REPO_DIR_KAGGLE = "/kaggle/working/scresia"

LOCAL_OUTPUT_DIR = "/content/friend_dmlpa_final_outputs"
KAGGLE_OUTPUT_DIR = "/kaggle/working/friend_dmlpa_final_outputs"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/scresia_results/friend_dmlpa_final_outputs"
USE_GOOGLE_DRIVE_OUTPUT = True

RUNTIME_PIP_PACKAGES = [
    "simpy>=4.1",
    "numpy>=1.26",
    "gymnasium>=0.29",
    "stable-baselines3>=2.3",
    "torch>=2.1",
    "einops>=0.7",
    "pandas>=2.2",
]

SEED = 42
FRAME_STACK = 30
FEATURES_DIM = 120

REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
STOCHASTIC_PT = True
STEP_SIZE_HOURS = 168.0

TOTAL_TIMESTEPS_DEBUG = 10_000
MAX_STEPS_DEBUG = 40
EVAL_EPISODES_DEBUG = 1
N_STEPS_DEBUG = 256
BATCH_SIZE_DEBUG = 64
N_EPOCHS_DEBUG = 2

TOTAL_TIMESTEPS_SERIOUS = 100_000
MAX_STEPS_SERIOUS = 260
EVAL_EPISODES_SERIOUS = 5
N_STEPS_SERIOUS = 1024
BATCH_SIZE_SERIOUS = 256
N_EPOCHS_SERIOUS = 10

if RUN_PROFILE == "debug":
    TOTAL_TIMESTEPS = TOTAL_TIMESTEPS_DEBUG
    MAX_STEPS = MAX_STEPS_DEBUG
    EVAL_EPISODES = EVAL_EPISODES_DEBUG
    N_STEPS = N_STEPS_DEBUG
    BATCH_SIZE = BATCH_SIZE_DEBUG
    N_EPOCHS = N_EPOCHS_DEBUG
elif RUN_PROFILE == "serious":
    TOTAL_TIMESTEPS = TOTAL_TIMESTEPS_SERIOUS
    MAX_STEPS = MAX_STEPS_SERIOUS
    EVAL_EPISODES = EVAL_EPISODES_SERIOUS
    N_STEPS = N_STEPS_SERIOUS
    BATCH_SIZE = BATCH_SIZE_SERIOUS
    N_EPOCHS = N_EPOCHS_SERIOUS
else:
    raise ValueError("RUN_PROFILE must be debug or serious")

print({
    "RUN_PROFILE": RUN_PROFILE,
    "CONTRACT_PROFILE": CONTRACT_PROFILE,
    "DMLPA_VARIANT": DMLPA_VARIANT,
    "TOTAL_TIMESTEPS": TOTAL_TIMESTEPS,
    "MAX_STEPS": MAX_STEPS,
    "EVAL_EPISODES": EVAL_EPISODES,
})


In [ ]:
# =======================================
# 1) Portable Colab/Kaggle repository setup
# =======================================

from __future__ import annotations

from collections import deque
from datetime import datetime, timezone
from pathlib import Path
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()
if not (IN_COLAB or IN_KAGGLE):
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {cmd}")
    return result

if IN_COLAB or IN_KAGGLE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])
else:
    print("local run: skipping pip install")

drive_mounted = False
if IN_COLAB and USE_GOOGLE_DRIVE_OUTPUT:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True, timeout_ms=120_000)
        drive_mounted = True
    except Exception as exc:
        print("WARNING: Drive mount failed; falling back to /content output.", repr(exc))

if IN_KAGGLE:
    REPO_DIR = Path(REPO_DIR_KAGGLE)
    output_root = Path(KAGGLE_OUTPUT_DIR)
elif IN_COLAB:
    REPO_DIR = Path(REPO_DIR_COLAB)
    output_root = Path(DRIVE_OUTPUT_DIR if drive_mounted else LOCAL_OUTPUT_DIR)
else:
    REPO_DIR = Path.cwd() if (Path.cwd() / "supply_chain").exists() else Path.cwd().parent
    output_root = REPO_DIR / "outputs" / "friend_dmlpa_final_outputs"

if IN_COLAB or IN_KAGGLE:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(REPO_DIR)])

sys.path.insert(0, str(REPO_DIR.parent))
sys.path.insert(0, str(REPO_DIR))
output_root.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_DIR)
print("outputs:", output_root)
print("python path head:", sys.path[:3])
run_cmd(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, check=False)


In [ ]:
# ==========================
# 2) Imports and env contract
# ==========================

import gymnasium as gym
import numpy as np
import pandas as pd
import torch
from einops import rearrange
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecNormalize

try:
    from scresia.supply_chain.external_env_interface import (
        get_episode_terminal_metrics,
        make_dkana_thesis_faithful_env,
    )
except ModuleNotFoundError:
    from supply_chain.external_env_interface import (
        get_episode_terminal_metrics,
        make_dkana_thesis_faithful_env,
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CONTRACT_KWARGS = {
    "scientific_thesis_factorized": dict(
        observation_version="v5",
        observation_mode="env_sdm_history_reward",
        action_space_mode="thesis_factorized",
        inventory_period_mode="thesis_strict",
        learn_initial_decision=True,
    ),
    "legacy_box18": dict(
        observation_mode="decision_reward",
        action_space_mode="onehot_18d",
        inventory_period_mode="thesis_strict",
        learn_initial_decision=False,
    ),
}
if CONTRACT_PROFILE not in CONTRACT_KWARGS:
    raise ValueError(f"Unknown CONTRACT_PROFILE={CONTRACT_PROFILE!r}")

ENV_KWARGS = dict(
    reward_mode=REWARD_MODE,
    risk_level=RISK_LEVEL,
    stochastic_pt=STOCHASTIC_PT,
    max_steps=MAX_STEPS,
    step_size_hours=STEP_SIZE_HOURS,
    **CONTRACT_KWARGS[CONTRACT_PROFILE],
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("ENV_KWARGS:", ENV_KWARGS)


In [ ]:
# ==============================================
# 3) Environment factory and contract smoke check
# ==============================================


def make_raw_env(seed=SEED):
    env = make_dkana_thesis_faithful_env(**ENV_KWARGS)
    env.reset(seed=seed)
    return env


def make_single_env(seed=SEED):
    def _init():
        env = make_dkana_thesis_faithful_env(**ENV_KWARGS)
        env.reset(seed=seed)
        return Monitor(env)
    return _init


def make_training_env(seed=SEED):
    venv = DummyVecEnv([make_single_env(seed)])
    venv = VecFrameStack(venv, n_stack=FRAME_STACK)
    venv = VecNormalize(venv, norm_reward=True, norm_obs=True)
    return venv

smoke_env = make_raw_env(SEED)
obs, info = smoke_env.reset(seed=SEED)
print("single observation shape:", obs.shape)
print("single action space:", smoke_env.action_space)
print("action_space_mode:", info.get("action_space_mode"))
print("inventory_period_mode:", info.get("inventory_period_mode"))
if CONTRACT_PROFILE == "scientific_thesis_factorized":
    assert smoke_env.action_space.nvec.tolist() == [6, 3]
else:
    assert smoke_env.action_space.shape == (18,)
smoke_env.close()

train_env = make_training_env(SEED)
stacked_obs = train_env.reset()
print("stacked observation shape:", stacked_obs.shape)
print("stacked observation space:", train_env.observation_space)
print("vector action space:", train_env.action_space)
assert train_env.observation_space.shape[0] % FRAME_STACK == 0


## DMLPA variants

`FriendDMLPAFaithful` is the original architecture lane. `FriendDMLPAPositionalV12` adds the sinusoidal positional encoding and LayerNorm from the new notebook. Use the positional version only as an ablation because it changes the architecture.


In [ ]:
# ==============================================
# 4) DMLPA feature extractors
# ==============================================

class FriendDMLPAFaithful(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, factor: int, features_dim: int = 120):
        super().__init__(observation_space, features_dim)
        flat_dim = int(np.prod(observation_space.shape))
        if flat_dim % factor != 0:
            raise ValueError(f"Observation dim {flat_dim} is not divisible by factor={factor}")
        self.obs_dimension = flat_dim // factor
        self.factor = int(factor)
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension, 100),
            torch.nn.GELU(),
            torch.nn.Linear(100, features_dim),
        )
        self.attlayer = torch.nn.TransformerEncoderLayer(
            d_model=features_dim,
            nhead=12,
            batch_first=True,
        )
        self.accumulated = torch.nn.TransformerEncoder(self.attlayer, num_layers=4)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        x = rearrange(observations, "b (d k) -> b d k", d=self.factor)
        x = self.latent_rw(x)
        x = self.accumulated(x)
        return x[:, -1, :]


class FriendDMLPAPositionalV12(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, factor: int, features_dim: int = 120):
        super().__init__(observation_space, features_dim)
        flat_dim = int(np.prod(observation_space.shape))
        if flat_dim % factor != 0:
            raise ValueError(f"Observation dim {flat_dim} is not divisible by factor={factor}")
        self.obs_dimension = flat_dim // factor
        self.factor = int(factor)
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension, 100),
            torch.nn.GELU(),
            torch.nn.Linear(100, features_dim),
        )
        self.pre_norm = torch.nn.LayerNorm(features_dim)
        self.attlayer = torch.nn.TransformerEncoderLayer(
            d_model=features_dim,
            nhead=12,
            batch_first=True,
        )
        self.accumulated = torch.nn.TransformerEncoder(self.attlayer, num_layers=4)
        self.register_buffer("pos_encoding", self._build_sinusoidal_pe(self.factor, features_dim))

    @staticmethod
    def _build_sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        x = rearrange(observations, "b (d k) -> b d k", d=self.factor)
        x = self.latent_rw(x)
        x = self.pre_norm(x + self.pos_encoding.to(dtype=x.dtype))
        x = self.accumulated(x)
        return x[:, -1, :]


EXTRACTOR_BY_VARIANT = {
    "faithful": FriendDMLPAFaithful,
    "positional_v1_2": FriendDMLPAPositionalV12,
}
ExtractorClass = EXTRACTOR_BY_VARIANT[DMLPA_VARIANT]

extractor = ExtractorClass(train_env.observation_space, factor=FRAME_STACK, features_dim=FEATURES_DIM)
with torch.no_grad():
    features = extractor(torch.as_tensor(stacked_obs, dtype=torch.float32))
print("extractor:", ExtractorClass.__name__)
print("DMLPA obs_dimension:", extractor.obs_dimension)
print("DMLPA feature shape:", tuple(features.shape))
assert features.shape[-1] == FEATURES_DIM


In [ ]:
# ==============================================
# 5) PPO with DMLPA feature extractor
# ==============================================

policy_kwargs = dict(
    features_extractor_class=ExtractorClass,
    features_extractor_kwargs=dict(features_dim=FEATURES_DIM, factor=FRAME_STACK),
    net_arch=dict(pi=[128], vf=[128]),
)

model = PPO(
    "MlpPolicy",
    train_env,
    policy_kwargs=policy_kwargs,
    device="auto",
    verbose=1,
    learning_rate=3e-4,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    seed=SEED,
)

start = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS)
elapsed_seconds = time.time() - start
print("training seconds:", elapsed_seconds)


In [ ]:
# ==============================================
# 6) Manual deterministic evaluation
# ==============================================


def normalize_stacked_obs(stacked: np.ndarray, vecnorm: VecNormalize) -> np.ndarray:
    normalized = (stacked - vecnorm.obs_rms.mean) / np.sqrt(vecnorm.obs_rms.var + vecnorm.epsilon)
    return np.clip(normalized, -vecnorm.clip_obs, vecnorm.clip_obs).astype(np.float32)


def evaluate_model(model: PPO, vecnorm: VecNormalize, n_episodes=EVAL_EPISODES) -> pd.DataFrame:
    rows = []
    for episode in range(n_episodes):
        env = make_raw_env(SEED + 10_000 + episode)
        obs, info = env.reset(seed=SEED + 10_000 + episode)
        frames = deque([np.zeros_like(obs, dtype=np.float32) for _ in range(FRAME_STACK - 1)] + [obs.astype(np.float32)], maxlen=FRAME_STACK)
        terminated = truncated = False
        total_reward = 0.0
        steps = 0
        while not (terminated or truncated):
            stacked = np.concatenate(list(frames), axis=0).reshape(1, -1)
            model_obs = normalize_stacked_obs(stacked, vecnorm)
            action, _ = model.predict(model_obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action[0])
            frames.append(obs.astype(np.float32))
            total_reward += float(reward)
            steps += 1
        terminal = get_episode_terminal_metrics(env)
        rows.append({
            "episode": episode,
            "steps": steps,
            "total_reward": total_reward,
            **terminal,
        })
        env.close()
    return pd.DataFrame(rows)

train_env.training = False
train_env.norm_reward = False
results = evaluate_model(model, train_env)
display(results)
print(results.mean(numeric_only=True))


In [ ]:
# ==============================================
# 7) Save artifacts
# ==============================================

run_label = f"{CONTRACT_PROFILE}_{DMLPA_VARIANT}_{RUN_PROFILE}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
run_dir = output_root / run_label
run_dir.mkdir(parents=True, exist_ok=True)

model_path = run_dir / "ppo_friend_dmlpa.zip"
vecnorm_path = run_dir / "vecnormalize.pkl"
results_path = run_dir / "evaluation_results.csv"
manifest_path = run_dir / "manifest.json"

model.save(str(model_path))
train_env.save(str(vecnorm_path))
results.to_csv(results_path, index=False)

manifest = {
    "notebook": "friend_dmlpa_minimal_colab_kaggle_final.ipynb",
    "run_profile": RUN_PROFILE,
    "contract_profile": CONTRACT_PROFILE,
    "dmlpa_variant": DMLPA_VARIANT,
    "seed": SEED,
    "frame_stack": FRAME_STACK,
    "features_dim": FEATURES_DIM,
    "reward_mode": REWARD_MODE,
    "risk_level": RISK_LEVEL,
    "stochastic_pt": STOCHASTIC_PT,
    "step_size_hours": STEP_SIZE_HOURS,
    "max_steps": MAX_STEPS,
    "total_timesteps": TOTAL_TIMESTEPS,
    "eval_episodes": EVAL_EPISODES,
    "training_seconds": elapsed_seconds,
    "env_kwargs": ENV_KWARGS,
    "model_path": str(model_path),
    "vecnormalize_path": str(vecnorm_path),
    "results_path": str(results_path),
    "mean_metrics": results.mean(numeric_only=True).to_dict(),
}
manifest_path.write_text(json.dumps(manifest, indent=2))

print("saved model:", model_path)
print("saved vecnormalize:", vecnorm_path)
print("saved results:", results_path)
print("saved manifest:", manifest_path)


## Interpretation rule

Use `faithful` as the friend's DMLPA baseline. Use `positional_v1_2` only to answer whether explicit temporal position helps. Do not report either as the central scientific contribution before the static ladder and PPO-vs-static action-space comparisons are clean.
